# Comparación de fragmentación documental

Tres construcciones bajo las mismas fuentes, preguntas, modelo, revisión y k. No selecciona una ganadora automáticamente. Requiere dependencias y descarga inicial del modelo; no necesita Groq ni Supabase.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import display
root = Path.cwd().resolve()
if not (root / "src").is_dir(): root = root.parent
sys.path.insert(0, str(root))
from src.rag.config import RAGConfig
from src.rag.ingestion import load_corpus
from src.rag.embeddings import SentenceEmbedder
from src.rag.comparison import compare_strategies


## Extraer y revisar fuentes
El manifiesto enumera exclusivamente los siete PDF simulados. La extracción conserva identificadores de sección y páginas y verifica hashes.

In [ ]:
sections = load_corpus(RAGConfig().manifest)
display([s.model_dump() for s in sections[:2]])
print("Secciones:", len(sections))


## Ejecutar o revisar una comparación guardada
Activar para generar embeddings reales. Por defecto muestra la evidencia guardada si existe; no finge una ejecución.

In [ ]:
RUN_COMPARISON = False
REVISION = None  # Usar una revisión exacta para reproducir una construcción.
output = root / "docs/rag_comparison.json"
if RUN_COMPARISON:
    embedder = SentenceEmbedder(REVISION)
    questions = json.loads((root / "cases/rag_questions.json").read_text())
    comparison = compare_strategies(sections, embedder, questions, root / "data/indexes", k=3)
    output.write_text(json.dumps(comparison, ensure_ascii=False, indent=2))
elif output.exists():
    comparison = json.loads(output.read_text())
else:
    comparison = None
    print("Sin comparación ejecutada; activa RUN_COMPARISON para generarla.")


In [ ]:
if comparison:
    display({name: {key: data[key] for key in ("corpus_id", "count", "tokens_min", "tokens_max", "tokens_mean", "exact_duplicate_chunks")} for name, data in comparison["strategies"].items()})
    for name, data in comparison["strategies"].items():
        print(name, "cortes reales:")
        display(data["chunks"][:3])
        display(data["trials"])


## Revisión y decisión

- Leer las condiciones completas y señalar cortes que separen requisitos y consecuencias.
- Examinar pasajes esperados en Top-k, duplicados y cobertura; un score no demuestra suficiencia.
- Revisar preguntas ambiguas, VPN fuera de cobertura y estados actuales.
- Registrar estrategia elegida y justificación en la bitácora; `winner` permanece vacío hasta esa revisión.
- Cada variante usa un `corpus_id` distinto; no mezclar sus vectores ni subir las tres como un mismo corpus.